# Lesson 4.5: Merging, Cleaning, and Reshaping

**Analyst:** John Ko
**Date:** 11/05/26
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Clean the marketing team's messy plan_metadata file,
merge it into the customer data, and produce the consolidated per-tier
summary table for Hannah's 2 PM checkpoint.

Friday checkpoint: 3 PM. Memo table to Hannah by 2 PM for review.

oad the customer data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-customers-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

print(f"Customer data: {df.shape}")
df.head(2)

Customer data: (200000, 16)


,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2025-10-17,Basic,9.99,2025-12-25,134,4.70,0,True,NaT
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-11-24,Pro,29.99,2025-12-14,236,3.40,0,True,NaT


## Inspecting the marketing team's plan metadata

Load it. Look at it. Document every problem before fixing anything.

In [2]:
plans_raw = pd.read_csv(
    'plan_metadata_raw.csv',
    parse_dates=['launch_date']
)
print(f"Shape: {plans_raw.shape}")
print()
plans_raw

Shape: (5, 5)



,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date
0,Basic,Starter Plan,Individual,9.99,2019-03-15
1,PRO,Professional,SMB,29.99,2020-08-01
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30
3,PRO,Professional,SMB,29.99,2020-08-01
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12


In [3]:
# Survey the problems
print("Unique plan_tier values (as-is):")
for value in plans_raw['plan_tier'].unique():
    print(f"  '{value}' (length {len(value)})")
print()

# Missing values
print("Missing values per column:")
print(plans_raw.isna().sum())

print()# Duplicates
print(f"Duplicate rows: {plans_raw.duplicated().sum()}")

Unique plan_tier values (as-is):
  ' Basic ' (length 7)
  'PRO' (length 3)
  ' enterprise ' (length 12)
  'Legacy' (length 6)

Missing values per column:
plan_tier             0
marketing_name        0
target_segment        0
monthly_fee_listed    1
launch_date           0
dtype: int64

Duplicate rows: 1


**Observation:** Five rows total, four distinct after deduplication.
Plan tier values have whitespace and capitalization mess that will prevent
the join column from matching the customer data. monthly_fee_listed is
missing for Legacy. Need to clean before merging.

In [4]:
# Strip whitespace, then title-case
plans_raw['plan_tier_clean'] = plans_raw['plan_tier'].str.strip().str.title()

print("Cleaned plan_tier values:")
for value in plans_raw['plan_tier_clean'].unique():
    print(f"  '{value}' (length {len(value)})")

Cleaned plan_tier values:
  'Basic' (length 5)
  'Pro' (length 3)
  'Enterprise' (length 10)
  'Legacy' (length 6)


In [5]:
# How many rows before
print(f"Rows before dedupe: {len(plans_raw)}")

# Drop duplicates on the cleaned column
plans = plans_raw.drop_duplicates(subset=['plan_tier_clean'], keep='first').copy()

print(f"Rows after dedupe: {len(plans)}")
plans

Rows before dedupe: 5
Rows after dedupe: 4


,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date,plan_tier_clean
0,Basic,Starter Plan,Individual,9.99,2019-03-15,Basic
1,PRO,Professional,SMB,29.99,2020-08-01,Pro
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30,Enterprise
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12,Legacy


In [6]:
# Confirm the NaN remains
print("monthly_fee_listed values:")
print(plans[['plan_tier_clean', 'monthly_fee_listed']])

monthly_fee_listed values:
  plan_tier_clean  monthly_fee_listed
0           Basic                9.99
1             Pro               29.99
2      Enterprise              199.00
4          Legacy                 NaN


**Observation:** Leaving the Legacy monthly_fee_listed as NaN rather than
filling. The customer data already has a per-customer monthly_fee, so the
listed price isn't critical for the join. If Hannah asks during the
readout, we can investigate.


In [7]:
print("Customer plan_tier values:")
print(df['plan_tier'].value_counts())

Customer plan_tier values:
plan_tier
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64


In [8]:
# Inner join: only rows where plan_tier_clean matches on both sides
df_inner = pd.merge(
    df,
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='inner'
)

print(f"Customer rows before merge: {len(df)}")
print(f"Customer rows after inner merge: {len(df_inner)}")
print(f"Columns added: {df_inner.shape[1] - df.shape[1]}")
print()
print("Tier counts in inner merge result:")
print(df_inner['plan_tier_clean'].value_counts())

Customer rows before merge: 200000
Customer rows after inner merge: 200000
Columns added: 6

Tier counts in inner merge result:
plan_tier_clean
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64


In [9]:
df_enriched = pd.merge(
    df,
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='left'
)

print(f"Shape after left merge: {df_enriched.shape}")
print(f"NaN counts in metadata columns:")
print(df_enriched[['marketing_name', 'target_segment', 'monthly_fee_listed', 'launch_date']].isna().sum())

Shape after left merge: (200000, 22)
NaN counts in metadata columns:
marketing_name        0
target_segment        0
monthly_fee_listed    0
launch_date           0
dtype: int64


In [10]:
df_outer = pd.merge(
    df[['customer_id', 'plan_tier']],   # narrowed for readability
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='outer'
)

print(f"Outer merge shape: {df_outer.shape}")
print()
print("Rows where customer_id is NaN (these are unmatched metadata rows):")
print(df_outer[df_outer['customer_id'].isna()])

Outer merge shape: (200001, 8)

Rows where customer_id is NaN (these are unmatched metadata rows):
       customer_id plan_tier_x plan_tier_y marketing_name target_segment  \
139771         NaN         NaN      Legacy  Heritage Plan     Individual   

        monthly_fee_listed launch_date plan_tier_clean  
139771                 NaN  2017-06-12          Legacy  


In [11]:
# Re-create the per-tier summary, now with marketing names
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

memo_summary = df_enriched.groupby(['plan_tier_clean', 'marketing_name', 'target_segment']).agg({
    'customer_id': 'count',
    'monthly_fee': 'sum',
    'active': churn_rate_pct,
    'total_sessions': 'mean'
}).round(2)

memo_summary.columns = ['customer_count', 'total_mrr', 'churn_rate_pct', 'avg_sessions']
memo_summary = memo_summary.reset_index()
memo_summary

,plan_tier_clean,marketing_name,target_segment,customer_count,total_mrr,churn_rate_pct,avg_sessions
0,Basic,Starter Plan,Individual,119881,1197611.19,19.93,172.81
1,Enterprise,Enterprise Plus,Enterprise,19890,3958110.00,19.88,173.46
2,Pro,Professional,SMB,60229,1806267.71,20.09,173.26


In [12]:
sessions_by_tier_and_status = df_enriched.pivot_table(
    index='marketing_name',
    columns='active',
    values='total_sessions',
    aggfunc='mean'
).round(1)

sessions_by_tier_and_status.columns = ['churned_avg_sessions', 'active_avg_sessions']
sessions_by_tier_and_status


,churned_avg_sessions,active_avg_sessions
marketing_name,,
Enterprise Plus,25.60,210.20
Professional,25.50,210.40
Starter Plan,25.50,209.50


In [13]:
memo_summary.to_csv('friday-memo-summary.csv', index=False)
sessions_by_tier_and_status.to_csv('friday-memo-sessions-comparison.csv')
print("Saved friday-memo-summary.csv and friday-memo-sessions-comparison.csv")

Saved friday-memo-summary.csv and friday-memo-sessions-comparison.csv


In [14]:
mrr_by_tier = df_enriched.groupby('marketing_name')['monthly_fee'].sum()
mrr_pct = (mrr_by_tier / mrr_by_tier.sum() * 100).round(1)
print("MRR percent by marketing tier:")
print(mrr_pct)


MRR percent by marketing tier:
marketing_name
Enterprise Plus   56.90
Professional      25.90
Starter Plan      17.20
Name: monthly_fee, dtype: float64


**AI Use Note:** Used Claude to confirm the per-group-percent idiom.
Output matched the obvious groupby pattern. Verified on FinFlow data.
Tier 1 disclosure.